In [1]:
from common_constants import SPLITS, MODELS_FOLDER
from sick_loader import SICKMergedDataset
from activations_loader import ActivationDataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch as t

device: t.device = t.device("cuda" if t.cuda.is_available() else "cpu")

Check the total number of elements in the sick merged dataset

In [ ]:
# all_sick_merged_elements = []
number_of_elements = 0
language = "en"

for split in SPLITS:
    sick_merged_dataset = SICKMergedDataset(language, split)
    for sick_merged_element in sick_merged_dataset:
        # print(sick_merged_element, number_of_elements)
        # all_sick_merged_elements.append(sick_merged_element)
        number_of_elements += 1

# print(all_sick_merged_elements)
print(number_of_elements)

9840


Check that the activations make sense

In [4]:
activations_dataset = ActivationDataset("es", "train", 0, "standard", "olmo_model")

for i in range(8):
    activations, labels = activations_dataset[i]
    print(f"{activations}\n\n{labels}\n-----------------")

tensor([1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 2, 0, 1, 1, 1, 0, 1, 1, 1, 1,
        1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 2, 2, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1,
        1, 1, 2, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 2, 1, 1, 2, 1, 2, 1, 0, 1, 1, 1,
        1, 1, 1, 1, 2, 2, 0, 2, 0, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 0, 1,
        1, 1, 2, 2, 1, 1, 1, 1, 0, 2, 1, 1, 0, 2, 1, 1, 2, 1, 1, 1, 1, 0, 1, 2,
        1, 0, 0, 1, 1, 0, 2, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 2, 1, 1, 1, 1,
        0, 0, 1, 1, 0, 0, 2, 1, 1, 1, 2, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 2, 1,
        2, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1,
        1, 1, 2, 1, 1, 1, 0, 1, 0, 1, 2, 0, 1, 1, 1, 2, 0, 0, 1, 1, 1, 1, 0, 2,
        1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 2, 0, 1, 1, 0, 2, 1, 1, 1, 1, 1, 2, 1, 0,
        1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 2, 1], dtype=torch.int32)
tensor([1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 2, 0,
        1, 1, 1, 1, 2, 1, 1, 0, 1, 0, 2, 1, 

In [ ]:
from itertools import combinations
from common_constants import LANGUAGES

list(combinations(LANGUAGES, 2))

In [18]:
def run_nli_inference(tokenizer, hf_model):
    """Run NLI task inference with any model."""
    sentence_tuple_batch = [
        (
            "Three boys are jumping in the leaves",
            "Three kids are jumping in the leaves",
        ),
        (
            "Two women are sparring in a kickboxing match",
            "Two people are kickboxing and spectators are not watching",
        ),
    ]

    prompts = [
        f"Premise: {sent_a}. Hypothesis: {sent_b}. Do these sentences entail, contradict, or are neutral to each other?"
        for sent_a, sent_b in sentence_tuple_batch
    ]

    tokens = tokenizer(prompts, return_tensors="pt", padding=True).to(device)

    with t.no_grad():
        generated_ids = hf_model.generate(
            **tokens, max_new_tokens=20, pad_token_id=tokenizer.pad_token_id
        )

    gen_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

    for i, text in enumerate(gen_text):
        print(f"Response {i+1}: {text}")

In [19]:
# model_name: str = "olmo_model"
model_name: str = "tiny_aya_global"
model_filepath: str = f"{MODELS_FOLDER}/{model_name}"

tokenizer = AutoTokenizer.from_pretrained(
    model_filepath,
    local_files_only=True,
    #   padding_side='left',
)
hf_model = AutoModelForCausalLM.from_pretrained(
    model_filepath, local_files_only=True
).to(device)  # type: ignore

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [20]:
run_nli_inference(tokenizer, hf_model)

Response 1: Premise: Three boys are jumping in the leaves. Hypothesis: Three kids are jumping in the leaves. Do these sentences entail, contradict, or are neutral to each other?
Neutral
The premise and hypothesis are the same, just rephrased. The premise is
Response 2: Premise: Two women are sparring in a kickboxing match. Hypothesis: Two people are kickboxing and spectators are not watching. Do these sentences entail, contradict, or are neutral to each other?
Neutral
Premise: A man is wearing a red shirt and a woman is wearing a
